# URJA Load — 1K RPS vs 40 Endpoints
Find bottleneck before Gumroad launch. Requires Latitude API running with `ngrok` or public URL.


In [ ]:
!pip -q install locust httpx --progress-bar off 2>&1 | tail -1
import httpx, asyncio, time
BASE="https://YOUR_NGROK_URL"  # ← replace with `ngrok http 8000` from Latitude
print("Set BASE to your ngrok URL, then run next cell")

In [ ]:
async def hit(path):
    async with httpx.AsyncClient() as c:
        r=await c.get(f"{BASE}{path}", headers={"Authorization":"Bearer mock"}, timeout=5)
        return r.status_code

import asyncio
paths=["/api/v1/telemetry/latest","/api/v1/health/alerts","/api/v1/carbon/portfolio","/api/v1/assets"]
async def burst(n=100):
    t0=time.time(); results=await asyncio.gather(*[hit(p) for p in paths*n])
    print(f"{len(results)} req in {time.time()-t0:.2f}s = {len(results)/(time.time()-t0):.0f} rps, 200s: {results.count(200)}")
await burst(25)  # 100 req
print("If <500 rps, check DB pool + ARQ workers")